In [7]:
import asyncio
import json

from openai import OpenAI

from fastmcp import Client as MCPClient
from fastmcp.client.transports import StreamableHttpTransport


# LiteLLM endpoint
llm = OpenAI(
    api_key="sk-aorPyWzQfmJpUdQHQ3dOrA",
    base_url="http://localhost:4000/v1"
)

# MCP server
transport = StreamableHttpTransport(
    url="http://localhost:8000/mcp"
)

mcp = MCPClient(transport)


async def chat():

    async with mcp:

        # Load MCP tools
        tool_list = await mcp.list_tools()

        # Convert to OpenAI tool schema
        tools = []

        for tool in tool_list:
            tools.append({
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema
                }
            })

        messages = [
            {
                "role": "user",
                "content": "What is the weather in Dhaka?"
            }
        ]

        # Ask model
        response = llm.chat.completions.create(
            model="local-gemma-4",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        messages.append(msg.model_dump())

        # Execute tool calls
        if msg.tool_calls:

            for tool_call in msg.tool_calls:

                name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                result = await mcp.call_tool(name, args)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(result)
                })

        # Final response
        final = llm.chat.completions.create(
            model="local-gemma-4",
            messages=messages
        )

        print(final.choices[0].message.content)


import nest_asyncio
nest_asyncio.apply()

asyncio.run(chat())

The current weather in Dhaka, Bangladesh is **Clear sky** with a temperature of **27.5°C** and a wind speed of **10.4 km/h**.
